In [ ]:
from dotenv import load_dotenv
from tavily import TavilyClient
import os
from langchain_core.messages import HumanMessage
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama, OllamaEmbeddings # NEW: Local Ollama
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda

load_dotenv()

True

## Rsearch pipeline

In [ ]:
def build_research_chain(user_query: str, search_type: str = "advanced", max_results: int = 5):
    
    search_type_map = {"basic": "basic", "advanced": "advanced", "comprehensive": "advanced"}
    client = TavilyClient(os.getenv("TAVILY_API_KEY"))
    response = client.search(
        query=user_query,
        search_depth=search_type_map.get(search_type, "advanced"),
        include_images=False,
        max_results=max_results,
        include_raw_content="text",
    )

    articles = response.get("results", [])

    print("--- Research Results ---")
    final_response = []
    content_response = []
    for i, article in enumerate(articles, 1):
        title = article.get("title")
        link  = article.get("url")
        content_markdown = article.get("raw_content") or article.get("content", "")
        print(f"{i}. {title}")
        print(f"   Link: {link}")
        final_response.append((title, link))
        content_response.append((title, content_markdown))

    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    all_texts = [text for _, text in content_response if text.strip()]
    metadatas = [
        {"title": title, "source": link}
        for (title, link), (_, text) in zip(final_response, content_response)
        if text.strip()
    ]
    chunks = splitter.create_documents(all_texts, metadatas=metadatas)
    print(f"Total chunks created: {len(chunks)}")

    # NEW: Local Embeddings via Ollama (Matches app.py)
    embeddings = OllamaEmbeddings(model="nomic-embed-text")
    vector_store = FAISS.from_documents(chunks, embeddings)

    base_retriever = vector_store.as_retriever(search_kwargs={"k": 5})

    # NEW: Local Gemma 2 via Ollama (Matches app.py)
    chat_model = ChatOllama(model="gemma2:2b", temperature=0.3)

    chat_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful assistant. Answer ONLY from the provided context. If the context is insufficient, just say you don't know."),
        ("human", "Context:\n{context}\n\nQuestion: {question}")
    ])

    def format_docs(retrieved_docs):
        if not retrieved_docs:
            return "No relevant content found."
        return "\n\n".join(f"Source: {doc.metadata.get('title')}\nContent: {doc.page_content}" for doc in retrieved_docs)

    parallel_chain = RunnableParallel({
        "context":  base_retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough(),
    })

    parser = StrOutputParser()
    main_chain = parallel_chain | chat_prompt | chat_model | parser

    return main_chain, final_response

## Usage — call with any topic
Re-run the cell below to switch topics. Each call builds a completely fresh FAISS store.
Then use the Q&A cell below it to ask as many questions as you want without rebuilding.

In [ ]:

user_query  = input("What topic do you want to research? ")
search_type = "advanced" 
max_results = 5

main_chain, articles = build_research_chain(user_query, search_type, max_results)

print("\nResearch Complete. You can now ask questions in the next cell.")

main_chain, articles = build_research_chain(user_query, search_type, max_results)

print("\nArticles fetched:")
for title, link in articles:
    print(f"  - {title}")
    print(f"    {link}")


--- Research Results ---
1. What Is Claude Mythos? Anthropic's Most Dangerous AI Model ...
   Link: https://www.mindstudio.ai/blog/what-is-claude-mythos-anthropic-most-dangerous-ai-model/
2. Claude Mythos Preview Is Everyone's Problem - The Atlantic
   Link: https://www.theatlantic.com/technology/2026/04/claude-mythos-hacking/686746/
3. Claude Mythos Is Amazing: Here's Why You'll Never Get to Use It
   Link: https://serverpartdeals.com/blogs/blog-posts/claude-mythos-is-amazing-heres-why-youll-never-get-to-use-it
4. Claude Mythos explained.. - YouTube
   Link: https://www.youtube.com/watch?v=ENB3lqbCup8
5. Exclusive: Anthropic 'Mythos' AI model representing 'step change' in ...
   Link: https://fortune.com/2026/03/26/anthropic-says-testing-mythos-powerful-new-ai-model-after-data-leak-reveals-its-existence-step-change-in-capabilities/
Total chunks created: 70

Articles fetched:
  - What Is Claude Mythos? Anthropic's Most Dangerous AI Model ...
    https://www.mindstudio.ai/blog/what-is-c

In [ ]:
while True:
    question = input("\nAsk a question about the articles (or type 'exit' to stop): ")
    if question.lower() == 'exit':
        break
    
    answer = main_chain.invoke(question)
    print(f"\nAI: {answer}")

Claude Mythos is an advanced AI model developed by Anthropic that has the capability to discover thousands of previously unknown security vulnerabilities, or zero-day vulnerabilities. Due to the significant risks associated with its potential misuse, Anthropic has decided not to release Claude Mythos to the public. The model was designed to push the boundaries of AI capabilities, but its findings were so concerning that it will remain unreleased.
